In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
data_path = "/content/drive/MyDrive/Colab Notebooks/amazon_sales_data 2025.csv"

In [57]:
import mlflow
import pandas as pd
import numpy as np
import os
import hashlib
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib

# Thiết lập experiment MLflow
mlflow.set_experiment("Amazon Sales Prediction")

# 1. DATA PREPROCESSING
def preprocess_data(data_path, features=None, target="Quantity", test_size=0.2, val_size=0.1, random_state=42):

    print(f"Loading data from: {data_path}")
    df = pd.read_csv(data_path)

    # Tính dataset version bằng hashing
    with open(data_path, 'rb') as f:
        dataset_bytes = f.read()
        data_hash = hashlib.md5(dataset_bytes).hexdigest()
        file_size = len(dataset_bytes)

    # Nếu features không được chỉ định, sử dụng tất cả trừ target và một số cột không liên quan
    if features is None:
        # Loại bỏ Order ID (ID đơn hàng không có ý nghĩa dự đoán)
        # Loại bỏ Customer Name (tên khách hàng không nên ảnh hưởng đến số lượng)
        # Vẫn giữ lại Total Sales vì nó có thể có tương quan với Quantity, nhưng cần cẩn thận
        # về data leakage trong trường hợp này
        exclude_columns = [target, 'Order ID', 'Customer Name']
        features = [col for col in df.columns if col not in exclude_columns]

    # Chuyển đổi cột Date sang định dạng datetime
    df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%y')

    # Trích xuất các thuộc tính thời gian có ý nghĩa
    if 'Date' in features:
        df['Day'] = df['Date'].dt.day
        df['Month'] = df['Date'].dt.month
        df['DayOfWeek'] = df['Date'].dt.dayofweek

        # Thêm các cột thời gian mới vào features
        features.extend(['Day', 'Month', 'DayOfWeek'])

        # Loại bỏ cột Date gốc khỏi features vì đã được trích xuất thành các feature khác
        features.remove('Date')

    # Xử lý categorical features
    categorical_features = df[features].select_dtypes(include=['object']).columns.tolist()
    numerical_features = [f for f in features if f not in categorical_features and f != 'Date']

    # Kiểm tra data leakage tiềm ẩn với Total Sales
    if 'Total Sales' in features and target == 'Quantity':
        print("Warning: 'Total Sales' có thể gây data leakage vì nó phụ thuộc trực tiếp vào 'Quantity'")
        print("Đề xuất: Có thể loại bỏ 'Total Sales' hoặc thay đổi mục tiêu dự đoán")

    # Trích xuất features và target
    X = df[features]
    y = df[target]

    # Chia thành train và temp (temp sẽ được chia thành val và test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(test_size + val_size), random_state=random_state
    )

    # Chia temp thành validation và test
    val_ratio = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - val_ratio), random_state=random_state
    )

    # Tính toán statistics cho dữ liệu
    data_stats = {
        "train_mean": y_train.mean(),
        "train_std": y_train.std(),
        "train_min": y_train.min(),
        "train_max": y_train.max(),
    }

    # Tạo và trả về metadata của dataset
    dataset_metadata = {
        "dataset_path": data_path,
        "dataset_version": data_hash,
        "dataset_size_bytes": file_size,
        "dataset_rows": len(df),
        "dataset_columns": list(df.columns),
        "dataset_features_used": features,
        "dataset_target": target,
        "dataset_description": f"Amazon Sales Data loaded at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "train_samples": len(X_train),
        "validation_samples": len(X_val),
        "test_samples": len(X_test),
        "categorical_features": categorical_features,
        "numerical_features": numerical_features,
        "data_statistics": data_stats
    }

    return {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "metadata": dataset_metadata
    }

# 2. MODEL TRAINING
def train_model(data, model_type="linear", hyperparams=None, model_dir="checkpoints"):

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_val = data["X_val"]
    y_val = data["y_val"]

    # Tạo thư mục lưu checkpoints nếu chưa tồn tại
    os.makedirs(model_dir, exist_ok=True)

    # Mặc định hyperparameters nếu không được cung cấp
    if hyperparams is None:
        if model_type == "linear":
            hyperparams = {"fit_intercept": True, "copy_X": True, "n_jobs": None}
        elif model_type == "ridge":
            hyperparams = {"alpha": 1.0, "fit_intercept": True, "max_iter": 1000}
        elif model_type == "lasso":
            hyperparams = {"alpha": 0.1, "fit_intercept": True, "max_iter": 1000}
        elif model_type == "random_forest":
            hyperparams = {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "random_state": 42}

    # Xác định categorical và numerical features
    categorical_features = data["metadata"]["categorical_features"]
    numerical_features = data["metadata"]["numerical_features"]

    # Tạo preprocessor cho pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='passthrough'
    )

    # Chọn mô hình dựa trên model_type
    if model_type == "linear":
        model = LinearRegression(**hyperparams)
    elif model_type == "ridge":
        model = Ridge(**hyperparams)
    elif model_type == "lasso":
        model = Lasso(**hyperparams)
    elif model_type == "random_forest":
        model = RandomForestRegressor(**hyperparams)
    else:
        raise ValueError(f"Không hỗ trợ model type: {model_type}")

    # Tạo pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Huấn luyện mô hình
    pipeline.fit(X_train, y_train)

    # Dự đoán trên tập validation
    y_val_pred = pipeline.predict(X_val)

    # Tính toán metrics
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_rmse = np.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # Cross validation trên tập training
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_mse = -cv_scores.mean()
    cv_mse_std = cv_scores.std()

    # Lưu visualization cho performance
    plot_path = os.path.join(model_dir, f"{model_type}_performance_plot.png")
    plt.figure(figsize=(10, 6))
    plt.scatter(y_val, y_val_pred, alpha=0.5)
    plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title(f'{model_type} Model: Actual vs Predicted Values')
    plt.savefig(plot_path)
    plt.close()

    # Nếu mô hình là RandomForest, lưu thêm feature importance
    feature_importance_path = None
    if model_type == "random_forest":
        # Lấy tên của tất cả các features sau khi được xử lý bởi preprocessor
        feature_importance_path = os.path.join(model_dir, f"{model_type}_feature_importance.png")

        # Get feature importances
        importances = pipeline.named_steps['model'].feature_importances_

        # Lấy các tên feature sau khi one-hot encoding
        if hasattr(pipeline.named_steps['preprocessor'], 'transformers_'):
            all_feature_names = []

            # Lấy danh sách features từ các transformer
            for name, trans, cols in pipeline.named_steps['preprocessor'].transformers_:
                if name == 'num':
                    all_feature_names.extend(numerical_features)
                elif name == 'cat':
                    # Đối với categorical features, phải lấy các tên sau khi one-hot encoding
                    if hasattr(trans, 'get_feature_names_out'):
                        # Với scikit-learn phiên bản mới
                        cat_features = trans.get_feature_names_out(categorical_features)
                        all_feature_names.extend(cat_features)
                    else:
                        # Fallback nếu không có phương thức get_feature_names_out
                        for col in categorical_features:
                            all_feature_names.append(f"{col}")

            # Nếu số lượng feature names không khớp với số lượng importances,
            # chỉ sử dụng chỉ số thay vì tên
            if len(all_feature_names) != len(importances):
                all_feature_names = [f"Feature {i}" for i in range(len(importances))]
        else:
            # Fallback nếu không thể lấy tên features
            all_feature_names = [f"Feature {i}" for i in range(len(importances))]

        # Sắp xếp theo importance giảm dần
        indices = np.argsort(importances)[::-1]

        # Lấy top features quan trọng nhất
        top_k = min(20, len(indices))  # Hiển thị tối đa 20 features để đảm bảo đồ thị dễ đọc

        plt.figure(figsize=(12, 8))
        plt.title('Top Feature Importances')
        plt.barh(range(top_k), importances[indices][:top_k], align='center')
        plt.yticks(range(top_k), [all_feature_names[i] for i in indices][:top_k])
        plt.xlabel('Relative Importance')
        plt.savefig(feature_importance_path)
        plt.close()

    # Lưu model checkpoint
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    checkpoint_path = os.path.join(model_dir, f"{model_type}_checkpoint_{timestamp}.joblib")
    joblib.dump(pipeline, checkpoint_path)

    # Tạo dictionary cho metrics
    metrics = {
        "validation_mse": val_mse,
        "validation_rmse": val_rmse,
        "validation_mae": val_mae,
        "validation_r2": val_r2,
        "cv_mse": cv_mse,
        "cv_mse_std": cv_mse_std
    }

    # Tạo dictionary cho artifacts
    artifacts = {
        "model_checkpoint": checkpoint_path,
        "performance_plot": plot_path
    }

    if feature_importance_path:
        artifacts["feature_importance_plot"] = feature_importance_path

    return pipeline, metrics, artifacts

# 3. MODEL EVALUATION
def evaluate_model(model, data):

    X_test = data["X_test"]
    y_test = data["y_test"]

    # Dự đoán trên tập test
    y_test_pred = model.predict(X_test)

    # Tính toán metrics
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Tính thêm các metrics hữu ích khác
    test_mape = np.mean(np.abs((y_test - y_test_pred) / (y_test + 1e-10))) * 100  # MAPE (Mean Absolute Percentage Error)

    # Trả về metrics
    return {
        "test_mse": test_mse,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "test_mape": test_mape
    }

# 4. MAIN TRAINING PIPELINE
def run_training_pipeline(data_path, model_type="linear", hyperparams=None, features=None):

    # Tạo run name có ý nghĩa
    run_name = f"{model_type.capitalize()}Regression_{datetime.now().strftime('%Y%m%d_%H%M')}"

    # Start MLflow run
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"Started MLflow run with ID: {run_id}")

        # 1. Preprocessing dữ liệu
        print("Preprocessing data...")
        data = preprocess_data(data_path, features=features)

        # Log dataset metadata
        print("Logging dataset metadata...")
        for key, value in data["metadata"].items():
            if isinstance(value, (str, int, float, bool)):
                mlflow.log_param(f"data_{key}", value)
            elif isinstance(value, (list, dict)):
                mlflow.log_param(f"data_{key}", str(value))

        # Log source code version
        mlflow.log_param("source_code_version", "1.0.0")

        # 2. Train model và log hyperparameters
        print(f"Training {model_type} model...")
        if hyperparams:
            for key, value in hyperparams.items():
                mlflow.log_param(key, value)

        model, training_metrics, artifacts = train_model(data, model_type, hyperparams)

        # Log training metrics
        print("Logging training metrics...")
        for key, value in training_metrics.items():
            mlflow.log_metric(key, value)

        # Log artifacts
        print("Logging artifacts...")
        for artifact_name, artifact_path in artifacts.items():
            mlflow.log_artifact(artifact_path)

        # 3. Evaluate model
        print("Evaluating model on test data...")
        test_metrics = evaluate_model(model, data)

        # Log test metrics
        print("Logging test metrics...")
        for key, value in test_metrics.items():
            mlflow.log_metric(key, value)

        # 4. Log model với signature
        print("Logging model...")
        mlflow.sklearn.log_model(
            model,
            "model",
            input_example=data["X_train"].iloc[:5],
            registered_model_name=f"amazon_sales_{model_type}_predictor"
        )

        # Kết thúc pipeline
        print(f"Training pipeline completed successfully!")
        print(f"Training metrics: {training_metrics}")
        print(f"Test metrics: {test_metrics}")
        print(f"MLflow run URL: {mlflow.get_artifact_uri()}")

        return model, training_metrics, test_metrics

# Chạy training pipeline
if __name__ == "__main__":
    # Đường dẫn đến dữ liệu
    data_path = "/content/drive/MyDrive/Colab Notebooks/amazon_sales_data 2025.csv"  # Điều chỉnh đường dẫn nếu cần

    # Chạy với tất cả các cột, loại bỏ một số cột không có ý nghĩa dự đoán
    # Chạy với LinearRegression
    linear_hyperparams = {
        "fit_intercept": True,
        "copy_X": True,
        "n_jobs": -1
    }

    print("Running LinearRegression pipeline with all features...")
    linear_model, linear_train_metrics, linear_test_metrics = run_training_pipeline(
        data_path,
        model_type="linear",
        hyperparams=linear_hyperparams,
        features=None  # None sẽ sử dụng tất cả các cột, ngoại trừ Order ID, Customer Name và target
    )

    # Chạy với RandomForest
    rf_hyperparams = {
        "n_estimators": 200,
        "max_depth": 15,
        "min_samples_split": 5,
        "random_state": 42,
        "n_jobs": -1
    }

    print("\nRunning RandomForest pipeline with all features...")
    rf_model, rf_train_metrics, rf_test_metrics = run_training_pipeline(
        data_path,
        model_type="random_forest",
        hyperparams=rf_hyperparams,
        features=None  # None sẽ sử dụng tất cả các cột, ngoại trừ Order ID, Customer Name và target
    )

    # Chạy với Ridge Regression
    ridge_hyperparams = {
        "alpha": 1.0,
        "fit_intercept": True,
        "max_iter": 1000,
        "solver": "auto",
        "random_state": 42
    }

    print("\nRunning Ridge Regression pipeline with all features...")
    ridge_model, ridge_train_metrics, ridge_test_metrics = run_training_pipeline(
        data_path,
        model_type="ridge",
        hyperparams=ridge_hyperparams,
        features=None
    )

    # So sánh kết quả
    print("\nModel Comparison:")
    print(f"LinearRegression Test RMSE: {linear_test_metrics['test_rmse']:.4f}")
    print(f"Ridge Regression Test RMSE: {ridge_test_metrics['test_rmse']:.4f}")
    print(f"RandomForest Test RMSE: {rf_test_metrics['test_rmse']:.4f}")

    print("\nModel Comparison (R²):")
    print(f"LinearRegression Test R²: {linear_test_metrics['test_r2']:.4f}")
    print(f"Ridge Regression Test R²: {ridge_test_metrics['test_r2']:.4f}")
    print(f"RandomForest Test R²: {rf_test_metrics['test_r2']:.4f}")

    # Kết luận về model tốt nhất
    models = {
        "LinearRegression": linear_test_metrics['test_rmse'],
        "Ridge Regression": ridge_test_metrics['test_rmse'],
        "RandomForest": rf_test_metrics['test_rmse']
    }

    best_model = min(models.items(), key=lambda x: x[1])[0]
    print(f"\nBest model based on Test RMSE: {best_model}")

Running LinearRegression pipeline with all features...
Started MLflow run with ID: 7b642bd2251141579ea95f003224c98d
Preprocessing data...
Loading data from: /content/drive/MyDrive/Colab Notebooks/amazon_sales_data 2025.csv
Đề xuất: Có thể loại bỏ 'Total Sales' hoặc thay đổi mục tiêu dự đoán
Logging dataset metadata...
Training linear model...
Logging training metrics...
Logging artifacts...
Evaluating model on test data...
Logging test metrics...
Logging model...


/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'amazon_sales_linear_predictor' already exists. Creating a new version of this model...
Created version '4' of model 'amazon_sales_linear_predictor'.


Training pipeline completed successfully!
Training metrics: {'validation_mse': 1.3152209191630484, 'validation_rmse': np.float64(1.1468308154052402), 'validation_mae': 0.9925827412499565, 'validation_r2': 0.29380320062121545, 'cv_mse': np.float64(1.4832170647260883), 'cv_mse_std': np.float64(0.3845600300234631)}
Test metrics: {'test_mse': 1.6087858640019777, 'test_rmse': np.float64(1.2683792272037482), 'test_mae': 1.0139275351976378, 'test_r2': 0.29077084198828074, 'test_mape': np.float64(51.86259464857667)}
MLflow run URL: file:///content/mlruns/652394929854252555/7b642bd2251141579ea95f003224c98d/artifacts

Running RandomForest pipeline with all features...
Started MLflow run with ID: edad0651819b4519a143b206b6cc0161
Preprocessing data...
Loading data from: /content/drive/MyDrive/Colab Notebooks/amazon_sales_data 2025.csv
Đề xuất: Có thể loại bỏ 'Total Sales' hoặc thay đổi mục tiêu dự đoán
Logging dataset metadata...
Training random_forest model...
Logging training metrics...
Logging 

/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'amazon_sales_random_forest_predictor' already exists. Creating a new version of this model...
Created version '3' of model 'amazon_sales_random_forest_predictor'.


Training pipeline completed successfully!
Training metrics: {'validation_mse': 0.1456673252313422, 'validation_rmse': np.float64(0.38166389039486326), 'validation_mae': 0.32490547619047616, 'validation_r2': 0.9217851561257827, 'cv_mse': np.float64(0.37349645689787103), 'cv_mse_std': np.float64(0.09971436947127016)}
Test metrics: {'test_mse': 0.21812503178380638, 'test_rmse': np.float64(0.46703857633369683), 'test_mae': 0.35906853903177427, 'test_r2': 0.9038401342932745, 'test_mape': np.float64(16.088663259658997)}
MLflow run URL: file:///content/mlruns/652394929854252555/edad0651819b4519a143b206b6cc0161/artifacts

Running Ridge Regression pipeline with all features...
Started MLflow run with ID: ca609a30902345c99028b7d59065bc08
Preprocessing data...
Loading data from: /content/drive/MyDrive/Colab Notebooks/amazon_sales_data 2025.csv
Đề xuất: Có thể loại bỏ 'Total Sales' hoặc thay đổi mục tiêu dự đoán
Logging dataset metadata...
Training ridge model...
Logging training metrics...
Loggin

/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Training pipeline completed successfully!
Training metrics: {'validation_mse': 1.297163444772267, 'validation_rmse': np.float64(1.1389308340598507), 'validation_mae': 0.9947325158268342, 'validation_r2': 0.3034990094650629, 'cv_mse': np.float64(1.4546132643025496), 'cv_mse_std': np.float64(0.36544573118351564)}
Test metrics: {'test_mse': 1.5419412575315812, 'test_rmse': np.float64(1.2417492732156445), 'test_mae': 0.9871497803444755, 'test_r2': 0.32023911680684025, 'test_mape': np.float64(50.13166219726302)}
MLflow run URL: file:///content/mlruns/652394929854252555/ca609a30902345c99028b7d59065bc08/artifacts

Model Comparison:
LinearRegression Test RMSE: 1.2684
Ridge Regression Test RMSE: 1.2417
RandomForest Test RMSE: 0.4670

Model Comparison (R²):
LinearRegression Test R²: 0.2908
Ridge Regression Test R²: 0.3202
RandomForest Test R²: 0.9038

Best model based on Test RMSE: RandomForest


Registered model 'amazon_sales_ridge_predictor' already exists. Creating a new version of this model...
Created version '2' of model 'amazon_sales_ridge_predictor'.


In [59]:
from pyngrok import ngrok

# Khởi động MLflow UI
get_ipython().system_raw("mlflow ui --host 0.0.0.0 &")

!ngrok authtoken 2vgiN9bHwACXbM7JhRhgH9WJ2Jv_6AUPNm7XjnS5jrS4aY6ry

# Disconnect existing ngrok tunnels before starting a new one
ngrok.kill() # this will kill all existing ngrok processes

# Tạo một tunnel công khai đến cổng 5000 (cổng mặc định của MLflow UI)
public_url = ngrok.connect(5000).public_url

print(f"MLflow UI đang chạy tại: {public_url}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
MLflow UI đang chạy tại: https://6c94-35-185-47-40.ngrok-free.app
